# 01_functions_demo — Functions, lambdas, decorators & generators

**Week 2 · Day 02** — Day 01 put logic **inline** in loops. Today you **name** that logic so the same pipeline is easier to read, test, and reuse.

**How to use this notebook**
- Run **top to bottom** with `Shift + Enter`.
- Each code cell is small on purpose — read the `#` comments, run, then check the `→` output.


---
## What you will cover

1. **Functions** — anatomy (`def`, parameters, `return`, …), defaults, `*args`, `**kwargs`.
2. **Lambdas** — tiny one-off functions, mostly for `key=` when sorting or picking a top row.
3. **Decorators** — wrap a function to add guards or extra checks **without** rewriting the function body.
4. **Generators** — `yield` rows or values lazily when lists would use too much memory.

All examples use **order / city / amount** data — the same shapes you cleaned on Pipeline Day 1.


---
## 0. Environment sanity check (30 seconds)

Verify the active Python environment before every run (same idea as picking the right database in SQL Workbench).


In [ ]:
import sys

py_version = sys.version.split()[0]
print(f"Python version: {py_version}")
print(f"Executable: {sys.executable}")


---
## 1. Functions — name the step, reuse it everywhere

A **function** is a named block of code with **inputs** (parameters) and an **output** (`return`). You call it by name instead of copying the same five lines into every loop.

### Why data engineers use functions

| Use case | Example in a pipeline |
|----------|------------------------|
| **Row transform** | Normalize `city`, cast types, parse timestamps |
| **Data quality** | `is_valid_row(row)` before writing to silver |
| **Path / key helpers** | Build `s3://bucket/orders/dt=2024-01-01/...` from one date |
| **Shared library** | Same `normalize_order()` in a notebook, Glue job, or Lambda |
| **Testing** | Unit-test one function with 3 dicts instead of running a full job |

In SQL you might stash logic in a **view** or **UDF**. In Python pipelines, a small `def` is usually the first version of that idea.

### Mental model (functions ↔ SQL)

| Python | SQL-style idea |
|--------|----------------|
| `def normalize_city(...)` | Reusable expression / scalar UDF |
| `return` value | Result of one `SELECT` column or aggregate |
| Default argument (`discount=0.0`) | `COALESCE(discount, 0)` |
| `*args` / `**kwargs` | Extra positional or named inputs when the count varies |

We keep function signatures **plain** in this lab (no type annotations) so you can focus on behavior first.


### Anatomy of a Python function

![Anatomy of a Python function](assets/anatomy-function.png)

**Regular functions** use the `def` keyword, can span **multiple lines**, and use **`return`** to send a value back to the caller.

| Label on diagram | Meaning |
|------------------|--------|
| **`def`** | Keyword that starts the definition |
| **Function name** | Descriptive `snake_case` name you call later |
| **Parameters** | Inputs in parentheses (`number`) |
| **`:`** | Starts the indented block |
| **Body** | Indented code that runs when called |
| **`return`** | Sends a value back; also exits the function |

Read the skeleton below once. Every pipeline step follows the same shape.

```text
def  function_name(parameter1, parameter2):
│    │             │
│    │             └─ names the function expects (placeholders)
│    └─ verb or action; use lowercase_with_underscores
└─ keyword that starts a function definition

    """One-line summary for humans (optional but good habit)."""
    │
    └─ docstring — what this step does in the pipeline

    result = parameter1 + parameter2   # body (indented) — the actual work
    return result                      # send a value back to whoever called you
```

| Element | Role | Data engineering analogy |
|---------|------|---------------------------|
| **`def`** | Starts the definition | Register a reusable step in your job |
| **Function name** | How you **call** the step later | e.g. `normalize_city`, `validate_row` |
| **Parameters** | Names on the **`def` line** (placeholders) | Template slots: `city`, `row` |
| **Arguments** | Values at **call** time | Actual data: `" hyd "`, one dict row |
| **Body** | Indented lines under `def` | Strip, filter, compute |
| **`return`** | Result to caller; ends the function | Clean string, `True`/`False`, etc. |
| **Docstring** | Triple-quoted string under `def` | Short README for the step |

**Parameters vs arguments (easy mix-up):**
- **Parameter** = name in the definition: `def f(city):`
- **Argument** = value at call time: `f("hyd")`

**`return` vs `print`:**
- `print` shows text on screen — the caller **does not** get that value back.
- `return` hands a Python value to the caller so you can assign it: `clean = normalize_city(" hyd ")`.


#### Try it — same example as the diagram (`calculate_square`)

**Step A:** `def` creates the function; the body does **not** run until you **call** it.


In [27]:
def calculate_square(number):       # def + name + parameter
    result = number ** 2            # body (indented)
    return result                   # return value to caller


print(f"function object     → {calculate_square}")  # defined, not run yet


function object     → <function calculate_square at 0x0000018F1BE4CE00>


**Step B:** Call with an **argument**; store whatever **`return`** sends back.


In [28]:
four = calculate_square(4)
nine = calculate_square(9)

print(f"calculate_square(4) → {four}")
print(f"calculate_square(9) → {nine}")


calculate_square(4) → 16
calculate_square(9) → 81


#### `return` vs `print` — only `return` feeds the next step


In [29]:
def show_total(amount):
    print(f"(debug) amount={amount}")   # visible in logs only
    return amount * 1.18                 # GST for downstream use


stored = show_total(100.0)
print(f"stored after return  → {stored}")


(debug) amount=100.0
stored after return  → 118.0


#### Early `return` — exit as soon as you know the answer

Data-quality functions often **return `False` immediately** when one rule fails (same idea as `continue` in a loop, but packaged as a function).


In [30]:
def has_amount(row):
    if row.get("amount") is None:
        return False          # stop here — no need to check further
    return row["amount"] > 0


print(f"missing amount       → {has_amount({'amount': None})}")
print(f"positive amount      → {has_amount({'amount': 10})}")


missing amount       → False
positive amount      → True


### Example — `normalize_city`: one string column, bronze → silver

Typical DE task: trim whitespace and standardize casing before load into a curated table.

Map to anatomy: **one parameter** `city`, **body** calls `.strip().upper()`, **`return`** sends the silver value to the caller.


In [3]:
def normalize_city(city):
    """Trim and uppercase — same rule as Day 01 `city` cleanup."""
    return city.strip().upper()


raw_city = " hyd "
print(f"input (argument)    → {raw_city!r}")
print(f"output (return)     → {normalize_city(raw_city)!r}")


input (argument)    → ' hyd '
output (return)     → 'HYD'


### Default **parameters** — optional inputs baked into the definition

Syntax: `def f(a, b=0.0):` — `b` is a **parameter** with a default. If the caller skips `b`, Python uses `0.0`.

Defaults are like `COALESCE(discount, 0)` — most rows have no discount, so you do not pass it every time.


In [ ]:
def line_revenue(units, unit_price, discount=0.0):
    # discount defaults to 0.0 when the caller does not pass it
    return units * unit_price - discount


print(f"line_revenue      → {line_revenue(7,24.5):.2f}")
print(f"with discount     → {line_revenue(7, 24.50, discount=10.0):.2f}")


line_revenue      → 171.50
with discount     → 161.50


### `*args` — extra **positional** values you do not name one-by-one

The `*` in the **parameter list** means: “collect any extra positional arguments into a **tuple**.”

**Data engineering idea:** sum revenue from **any number** of orders returned from a micro-batch — you might have 3 amounts today and 300 tomorrow, same function.


In [5]:
def sum_revenue(*amounts):
    # amounts is a tuple, e.g. (120.0, 55.5, 310.0)
    print(f"received tuple    → {amounts}")
    return sum(amounts)


day1 = sum_revenue(120.0, 55.5)
print(f"day1 total        → {day1:.2f}")

day2 = sum_revenue(10.0, 20.0, 30.0, 40.0)
print(f"day2 total        → {day2:.2f}")

# Unpack a list with * when calling
amounts_from_query = [120.0, 310.0]
print(f"from list         → {sum_revenue(*amounts_from_query):.2f}")


received tuple    → (120.0, 55.5)
day1 total        → 175.50
received tuple    → (10.0, 20.0, 30.0, 40.0)
day2 total        → 100.00
received tuple    → (120.0, 310.0)
from list         → 430.00


### `**kwargs` — extra **named** options (keyword arguments)

The `**` in the parameter list means: “collect any extra `name=value` pairs into a **dict**.”

**Data engineering idea:** build an object path or log line when **partition columns** change per job (`dt=...`, `status=...`, `region=...`) without rewriting the function each time.


In [65]:
a= {"abc":10, "def":20, "ghi":30}
# print("-".join(str(num) for num in a.items()))
print([f"{name}={num}" for name, num in a.items()])

['abc=10', 'def=20', 'ghi=30']


In [66]:
def build_object_key(table, **partitions):
    # partitions might be {"dt": "2024-01-01", "status": "completed"}
    print(partitions)
    parts = "/".join(f"{k}={v}" for k, v in partitions.items())
    return  f"{table}/{parts}/data.json"


key_a = build_object_key("orders", dt="2024-01-01", status="pending", part_no=1, abc= 4, st=131)
print(f"key_a             → {key_a}")

# key_b = build_object_key("orders", dt="2024-01-02", region="ap-south-1")
# print(f"key_b             → {key_b}")


# def log_event(step_name, **fields):
#     # Merge fixed step name with any extra columns from the row
#     bits = [f"{k}={v}" for k, v in fields.items()]
#     print(f"[{step_name}] " + ", ".join(bits))


# log_event("silver_write", order_id=1, city="HYD", amount=120.0)
# log_event("silver_write", order_id=6, city="DELHI", amount=310.0, note="priority")


{'dt': '2024-01-01', 'status': 'pending', 'part_no': 1, 'abc': 4, 'st': 131}
key_a             → orders/dt=2024-01-01/status=pending/part_no=1/abc=4/st=131/data.json


### Mixing normal parameters, `*args`, and `**kwargs`

Order in the definition: **required names first**, then `*args`, then `**kwargs` (you will see this pattern inside decorator `wrapper` functions in Section 3).


In [ ]:
def pipeline_footer(run_id, *steps, **meta):
    print(f"run_id            → {run_id}")
    print(f"steps (*args)     → {steps}")
    print(f"meta (**kwargs)   → {meta}")


pipeline_footer(
    "job-42",
    "extract",
    "transform",
    "load",
    owner="data-team",
    env="dev",
)


In [55]:
tuple1 = (100,50)

print(list(tuple1))

[100, 50]


---
## 2. Lambda expressions & `key=`

### Anatomy of a Python lambda

![Anatomy of a Python lambda](assets/anatomy-lambda.png)

**Lambda functions** are short, **inline**, **anonymous** functions. They contain **one expression** only; that expression is **returned automatically** (no `return` keyword).

| Label on diagram | Meaning |
|------------------|--------|
| **Variable** | Stores the lambda (`square = ...`) |
| **`lambda`** | Keyword that starts the anonymous function |
| **Parameter(s)** | Inputs before the `:` (no parentheses required) |
| **`:`** | Separates parameters from the expression |
| **Expression** | Single line of work; result is returned |


#### Same math as `calculate_square` — first as `def`, then as `lambda`


In [ ]:

def calculate_square(number):       # def + name + parameter
    result = number ** 2            # body (indented)
    return result                   # return value to caller

    
# lambda version (matches the diagram)
square = lambda number: number ** 2

print(f"square(5)           → {square(5)}")


square(5)           → 25


#### Pipeline-style use — `key=` when sorting (still one simple expression)

A **lambda** is a **small anonymous function** in one expression: `lambda r: r["amount"]` means “given row `r`, return `amount`.”

### Where lambdas show up in data engineering

| Situation | Why lambda is handy |
|-----------|---------------------|
| **`sorted(rows, key=...)`** | Preview “top N by revenue” in a notebook before SQL |
| **`max` / `min` with `key=`** | Pick the largest order in a batch for alerting |
| **Quick transforms** | One-liner when the logic is trivial and used once |

**When to prefer a normal `def`:** multi-line logic, reused in many files, or anything you will unit-test. Lambdas are for **short `key=` helpers**, not whole ETL steps.

| Pattern | SQL-ish idea |
|---------|-------------|
| `sorted(rows, key=lambda r: r["amount"])` | `ORDER BY amount` |
| `max(rows, key=lambda r: r["amount"])` | Top row by column (window-style pick) |


### Load a small `ORDERS` batch for sort / max demos


In [13]:
ORDERS = [
    {"order_id": 10, "amount": 50.0, "user_id": "u3"},
    {"order_id": 11, "amount": 250.0, "user_id": "u1"},
    {"order_id": 12, "amount": 200.0, "user_id": "u2"},
]

print(f"rows loaded       → {len(ORDERS)}")


rows loaded       → 3


### `sorted(..., key=lambda ...)` — order rows in memory


In [57]:
by_amount = sorted(ORDERS, key=lambda r: r["amount"])
# print(by_amount)
print(f"sorted by amount  → ids = {[r['order_id'] for r in by_amount]}")


sorted by amount  → ids = [10, 12, 11]


### `max(..., key=lambda ...)` — single “winner” row (e.g. fraud / SLA alert)


In [15]:
top = max(ORDERS, key=lambda r: r["amount"])
print(f"max amount order  → order_id={top['order_id']}, amount={top['amount']}")


max amount order  → order_id=11, amount=250.0


### Named `def` as `key=` — same behavior, easier to test

Production pipelines often use a named function when the sort key might grow or get reused.


In [16]:
def amount(row):
    return row["amount"]


by_amount_named = sorted(ORDERS, key=amount)
print(f"named key fn      → ids = {[r['order_id'] for r in by_amount_named]}")


named key fn      → ids = [10, 12, 11]


### Filtering without lambda

For `if` conditions in comprehensions, a plain expression is usually clearer than `lambda`.


In [17]:
big_amounts = [r for r in ORDERS if r["amount"] > 100]
print(f"amount > 100      → ids = {[r['order_id'] for r in big_amounts]}")


amount > 100      → ids = [11, 12]


---
## 3. Decorators — extra code around a function

### What is a decorator?

A **decorator** is a callable (usually a function) that takes **another function** as input and returns a **wrapper** function. When you call the decorated function, Python actually calls the wrapper, which can run code **before** and **after** the original logic. You attach a decorator with **`@name`** on the line directly above `def`.

Think of it as **packaging** a pipeline step: the step still does its job; the decorator adds a label (logging, timing, auth) on the outside without editing the step’s body.

### Where decorators are typically used

| Area | Typical use |
|------|-------------|
| **Web / APIs** | Register routes (`@app.route`), require login, rate limits |
| **Observability** | Log arguments and return values, measure duration, OpenTelemetry spans |
| **Resilience** | Retry failed calls, circuit-breaker wrappers |
| **Framework hooks** | `@classmethod`, `@staticmethod`, test fixtures (`@pytest.fixture`) |
| **Data jobs** | Time an ETL function, log row counts, validate config before `main()` runs |

### Anatomy of a Python decorator

![Anatomy of a Python decorator](assets/anatomy-decorator.jpg)

A **decorator** wraps another function: run something **before** and **after** `func()`, without changing `func` inside.

| Piece | Role |
|--------|------|
| `@name` above `def` | Attaches the wrapper |
| `wrapper` | Calls the original function in the middle |


#### Try it — same pattern as the diagram (`say_hello`)


In [68]:
def my_decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper


@my_decorator
def say_hello():
    print("Hello from the pipeline lab!")


say_hello()


Before
Hello from the pipeline lab!
After


In [77]:
# --- Problem: Skip bad order amounts with a decorator ---
#
# You have `double_amount(amount)` that returns amount * 2 for a report.
# If amount <= 0, you should not run that math — print a skip line instead.
#
# Write decorator `skip_if_invalid(fn)`:
#   1. Inside it, define `wrapper(amount)`.
#   2. If amount <= 0: print  skip invalid amount=<amount>  and return None.
#   3. Else: return fn(amount).
#   4. Return `wrapper` from `skip_if_invalid`.
#
# Define `double_amount(amount)`, put @skip_if_invalid on it, then call:
#   double_amount(-5)
#   double_amount(50)
#
# Write your code below:

def skip_if_invalid(double_amount):
    def wrapper(amount):
        if amount < 0 :
            print("Invalid amount")
            return None
        return double_amount(amount)
    return wrapper



@skip_if_invalid
def abc(amount):
    return amount * 2

@skip_if_invalid
def abc2(amount):
    return amount*3

# abc2(-100)

print(abc(100))
print(abc2(-100))

200
Invalid amount
None


#### Another try — shape the **return value**

First problem: **block** bad calls. This one: always run the inner function, but **change what the caller gets back** (here: money rounded to cents).


In [87]:
# --- Problem: Round totals without touching the pricing function ---
#
# `with_tax(amount)` returns amount * 1.18 (raw float — can show long decimals).
# Reporting wants 2 decimal places on every money function, without copy-pasting round() inside each one.
#
# Write decorator `round_to_cents(fn)`:
#   1. Define `wrapper(amount)`.
#   2. Call the inner function: value = fn(amount).
#   3. Return round(value, 2) to the caller.
#   4. Return `wrapper` from `round_to_cents`.
#
# Decorate `with_tax`, then call:  with_tax(10)
# (Compare mentally to problem 1: there you sometimes skip calling fn; here you always call fn.)
#
# Write your code below:

def round_to_cents(fn):
    def innerfunc(*xyz):
        print(xyz)
        return round(fn(*xyz),2)
    return innerfunc



@round_to_cents
def with_tax(amount, amount2):
    return amount * amount2


@round_to_cents
def with_5per_tax(amount):
    return amount * 1.05


print(with_5per_tax(1.78))

(1.78,)
1.87


---
## 4. Generators — yield one value at a time

### What is a generator?

A **generator** is a function that uses **`yield`** to produce values **one at a time** instead of building a full list and returning it. Each `yield` sends one value to the caller and **pauses** until the next item is requested. A **generator expression** looks like `(x for x in items)` — parentheses instead of square brackets.

Generators are **lazy**: code after a `yield` does not run until something asks for the next value (a `for` loop or `next()`).

### Anatomy of a Python generator

![Anatomy of a Python generator](assets/anatomy-generator.jpg)

| Label on diagram | Meaning |
|------------------|--------|
| **Generator function** | Looks like `def`, but `yield` makes it return a generator object when called |
| **`yield`** | Pauses, sends one value out, **saves local state** (e.g. `count`) |
| **Resume point** | Next `next()` or loop step continues **after** the `yield`, not from the top |
| **Generator object** | Created by `count_up_to(3)` — the body has **not** run yet |
| **`next()`** | Runs until the next `yield` and returns that value |

Concept from the diagram: generators hand back an **iterator** that produces one value at a time and pauses between steps — memory-friendly for large sequences.

### Where generators are typically used

| Use case | Why generators help |
|----------|---------------------|
| **Large files** | Read CSV/JSON line-by-line without loading the whole file into RAM |
| **Streaming ETL** | Map/filter rows from a DB cursor, API page, or S3 object as they arrive |
| **Long sequences** | Dates, IDs, or pages produced on demand |
| **Pipeline stages** | Parse → validate → emit one record at a time |

In data engineering, generators are the in-process cousin of **streaming**.


In [91]:
ORDERS = [
    {"order_id": 10, "amount": -50.0, "user_id": "u3"},
    {"order_id": 11, "amount": 250.0, "user_id": "u1"},
    {"order_id": 12, "amount": 200.0, "user_id": "u2"},
]

In [100]:
# --- Problem: Yield good amounts one at a time ---
#
# `ORDERS` is a list of dicts with an "amount" key .
#
# Write `good_amounts(orders)`:
#   - loop over each order
#   - if amount > 0, yield that amount
#
# Under the function, loop:  for amt in good_amounts(ORDERS): print(amt)
# (Do not use [ ... ] to collect everything into a list first.)

def good_amounts(orders):
    for order in orders:
        if order["amount"] > 0 :
            yield order["amount"]


for amt in (good_amounts(ORDERS)):
    print(amt)

# for amt in good_amounts(ORDERS):
#     print(amt)






250.0
200.0


#### list vs generator (type & size)

Same **logic** as `good_amounts`, but on a **~200k-row** fake bronze column (`BIG_BATCH_AMOUNTS`, ~1.6 MB as a list). Run once — it takes a few seconds.

`sys.getsizeof` measures the **list or generator object** (the list includes its pointer array). The gap vs a tiny generator is what matters


In [97]:
import sys
from itertools import islice

ROW_COUNT = 200_000  

BIG_BATCH_AMOUNTS = [float((i % 5000) + 1) for i in range(ROW_COUNT)]
# print(BIG_BATCH_AMOUNTS)

def good_amounts(amounts):
    for amount in amounts:
        if amount > 0:
            yield amount


amounts_list = [a for a in BIG_BATCH_AMOUNTS if a > 0]
amounts_gen = good_amounts(BIG_BATCH_AMOUNTS)

print(f"bronze rows (BIG_BATCH_AMOUNTS) → {len(BIG_BATCH_AMOUNTS):,}")
print(f"list type                       → {type(amounts_list).__name__}")
print(f"generator type                  → {type(amounts_gen).__name__}")
print(f"list size (bytes)               → {sys.getsizeof(amounts_list):,}")
print(f"generator size (bytes)          → {sys.getsizeof(amounts_gen):,}")
# print(f"filtered row count              → {len(amounts_list):,}")
print("first 3 from list               →", amounts_list[:3])
print(
    "first 3 from generator          →",
    list(islice(good_amounts(BIG_BATCH_AMOUNTS), 3)),
)


bronze rows (BIG_BATCH_AMOUNTS) → 200,000
list type                       → list
generator type                  → generator
list size (bytes)               → 1,624,056
generator size (bytes)          → 200
first 3 from list               → [1.0, 2.0, 3.0]
first 3 from generator          → [1.0, 2.0, 3.0]
